In [7]:
import requests
import pandas as pd
import time
import os
from tqdm import tqdm
import time

In [1]:
with open("../data/raw/app_ids_large_seed.txt", "r") as file:
    app_ids = [int(line.strip()) for line in file if line.strip()]

print(len(app_ids))
print(app_ids[:10])

483
[10, 20, 30, 40, 50, 60, 70, 80, 130, 220]


In [4]:
url = f"https://store.steampowered.com/appreviews/{730}"
response = requests.get(url, params = {
        "json": 1,
        "filter": "summary",
    }, timeout=15)
data = response.json()
print(data.keys())

dict_keys(['success', 'query_summary', 'reviews', 'cursor'])


In [5]:
def get_review_summary(app_id):
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    
    params = {
        "json": 1,
        "filter": "summary",
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
        
        query_summary = data.get("query_summary", {})
        
        total_positive = query_summary.get("total_positive")
        total_negative = query_summary.get("total_negative")
        
        total_reviews = None
        positive_ratio = None
        
        if total_positive is not None and total_negative is not None:
            total_reviews = total_positive + total_negative
            if total_reviews > 0:
                positive_ratio = total_positive / total_reviews
        
        return {
            "app_id": app_id,
            "review_score_desc": query_summary.get("review_score_desc"),
            "review_score": query_summary.get("review_score"),
            "total_positive": total_positive,
            "total_negative": total_negative,
            "total_reviews": total_reviews,
            "positive_ratio": positive_ratio
        }
    
    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return None

In [8]:
review_records = []

for app_id in tqdm(app_ids, desc="Collecting review data"):
    row = get_review_summary(app_id)
    if row is not None:
        review_records.append(row)
    time.sleep(1)

df_reviews = pd.DataFrame(review_records)
df_reviews.head()

Error for app_id 244210: Unexpected UTF-8 BOM (decode using utf-8-sig): line 1 column 1 (char 0)


,app_id,review_score_desc,review_score,total_positive,total_negative,total_reviews,positive_ratio
0,10,Overwhelmingly Positive,9,33453,1161,34614,0.966459
1,20,Very Positive,8,3617,575,4192,0.862834
2,30,Very Positive,8,1902,240,2142,0.887955
3,40,Very Positive,8,1043,219,1262,0.826466
4,50,Overwhelmingly Positive,9,12496,608,13104,0.953602


In [9]:
print(df_reviews.shape)
print(df_reviews.info())
print(df_reviews.isnull().sum())

(482, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482 entries, 0 to 481
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   app_id             482 non-null    int64  
 1   review_score_desc  482 non-null    object 
 2   review_score       482 non-null    int64  
 3   total_positive     482 non-null    int64  
 4   total_negative     482 non-null    int64  
 5   total_reviews      482 non-null    int64  
 6   positive_ratio     458 non-null    float64
dtypes: float64(1), int64(5), object(1)
memory usage: 26.5+ KB
None
app_id                0
review_score_desc     0
review_score          0
total_positive        0
total_negative        0
total_reviews         0
positive_ratio       24
dtype: int64


In [10]:
os.makedirs("../data/raw", exist_ok=True)
df_reviews.to_csv("../data/raw/steam_reviews_batch1.csv", index=False)

print("Saved raw reviews batch.")

Saved raw reviews batch.


In [11]:
df_reviews_clean = df_reviews.copy()

df_reviews_clean.columns = df_reviews_clean.columns.str.strip().str.lower()
df_reviews_clean = df_reviews_clean.drop_duplicates(subset="app_id")

df_reviews_clean["review_score"] = pd.to_numeric(df_reviews_clean["review_score"], errors="coerce")
df_reviews_clean["total_positive"] = pd.to_numeric(df_reviews_clean["total_positive"], errors="coerce")
df_reviews_clean["total_negative"] = pd.to_numeric(df_reviews_clean["total_negative"], errors="coerce")
df_reviews_clean["total_reviews"] = pd.to_numeric(df_reviews_clean["total_reviews"], errors="coerce")
df_reviews_clean["positive_ratio"] = pd.to_numeric(df_reviews_clean["positive_ratio"], errors="coerce")

os.makedirs("../data/cleaned", exist_ok=True)
df_reviews_clean.to_csv("../data/cleaned/steam_reviews_clean.csv", index=False)

print("Saved cleaned reviews.")

Saved cleaned reviews.
